# `split_window()` — `du-2015`

Estimar LST por Du usando o par térmico e emissividades prontas.

**Cadeia deste exemplo:** BT10/BT11 + ε10/ε11 + QA → Du geral → LST.

Este notebook executa uma etapa sobre arquivos já preparados. **Se você tem apenas os arquivos originais Level-1, prepare as entradas indicadas na tabela antes de executar este método.**

Execute apenas os **quatro blocos de código**, de cima para baixo. Os blocos ao final, em “Consulta”, são alternativas para leitura, e não uma continuação obrigatória.


## Antes de executar: separe estes arquivos

| Campo a preencher no passo 1 | Arquivo que você deve escolher | Unidade / tipo | Como obter se ainda falta |
| --- | --- | --- | --- |
| `e10_path` | Emissividade B10; por exemplo, `emissividade_B10.tif` | Adimensional, (0,1] | [Rongali](emissivity__rongali-2018.ipynb), executar uma vez para B10 e outra para B11 |
| `e11_path` | Emissividade B11; por exemplo, `emissividade_B11.tif` | Adimensional, (0,1] | [Rongali](emissivity__rongali-2018.ipynb), executar uma vez para B10 e outra para B11 |
| `bt10_path` | Temperatura de brilho B10; por exemplo, `BT_B10_K.tif` | K | [Temperatura de brilho](brightness.ipynb) com `banda=10` |
| `bt11_path` | Temperatura de brilho B11; por exemplo, `BT_B11_K.tif` | K | [Temperatura de brilho](brightness.ipynb) com `banda=11` |
| `qa_pixel_path` | Qualidade dos pixels; por exemplo, `PRODUTO_QA_PIXEL.TIF` | Bits inteiros, sem calibração | Arquivo original QA_PIXEL da mesma cena e na mesma grade |
| `qa_radsat_path` | Saturação/qualidade geométrica; por exemplo, `PRODUTO_QA_RADSAT.TIF` | Bits inteiros, sem calibração | Arquivo original QA_RADSAT da mesma cena e na mesma grade |

Todos os GeoTIFFs devem representar a mesma cena e ter dimensões, CRS e transformação idênticos. Para recortes, use a mesma janela nas QA. As unidades são verificadas pelo usuário: a leitura do arquivo não comprova sua grandeza física.

**Configuração desta execução:** O caminho principal usa coeficientes gerais, sem arquivo PWV. A variante com PWV está nas alternativas; é outra escolha de execução. As BT entram em K.

**Ambiente:** instale `python -m pip install -e ".[examples]"` na pasta da biblioteca e selecione o kernel desse ambiente no VS Code/Jupyter.


## Passo 1 — Preencher caminhos e escolher a saída

Substitua cada `CAMINHO_...` pelo caminho completo do seu arquivo, mantendo `Path(...)`. No Windows, escreva, por exemplo, `Path(r"C:\dados\radiancia_B4.tif")`. Preencha todos os campos da tabela, inclusive as duas QA.

O caminho principal usa coeficientes gerais, sem arquivo PWV. A variante com PWV está nas alternativas; é outra escolha de execução. As BT entram em K.

Ajuste `saida` para a pasta desejada. Caminhos relativos partem da pasta de trabalho do kernel; a célula imprime o caminho absoluto. Em nova execução com o mesmo destino, o GeoTIFF é substituído.


In [ ]:
from pathlib import Path
import numpy as np
import rasterio

# EDITE OS CAMINHOS E AS CONFIGURAÇÕES ABAIXO; mantenha o método deste notebook.
from pylstemp import read_quality_mask
from pylstemp import split_window

e10_path = Path("CAMINHO_DA_EMISSIVIDADE_B10.tif")
e11_path = Path("CAMINHO_DA_EMISSIVIDADE_B11.tif")
bt10_path = Path("CAMINHO_DA_BT10_K.tif")
bt11_path = Path("CAMINHO_DA_BT11_K.tif")
qa_pixel_path = Path("CAMINHO_DO_QA_PIXEL.tif")
qa_radsat_path = Path("CAMINHO_DO_QA_RADSAT.tif")

unidade = "kelvin"  # ou "celsius"; as BT entram sempre em K
diagnosticos = False
saida = Path("resultados/split_window__du-2015")

# Confira os arquivos antes de passar ao passo 2.
entradas_obrigatorias = {
    "e10_path": e10_path,
    "e11_path": e11_path,
    "bt10_path": bt10_path,
    "bt11_path": bt11_path,
    "qa_pixel_path": qa_pixel_path,
    "qa_radsat_path": qa_radsat_path,
}
pendentes = [f"{nome}: {caminho}" for nome, caminho in entradas_obrigatorias.items()
             if not caminho.is_file()]
if pendentes:
    raise FileNotFoundError("Preencha os caminhos de entrada com arquivos existentes:\n"
                            + "\n".join(pendentes))
print("Arquivos encontrados. Próximo passo: ler os dados e aplicar QA.")
print("Pasta de saída:", saida.resolve())


## Passo 2 — Ler as entradas, conferir a grade e aplicar QA

Execute esta célula depois de confirmar os caminhos. Ela carrega os dados, confere seu alinhamento, lê QA_PIXEL/QA_RADSAT com `read_quality_mask()` e combina QA com nodata em `mascara`. `True` indica pixel excluído. A leitura não recalibra dados prontos.

Água é preservada na política QA deste exemplo (`exclude_water=False`). A célula imprime as dimensões e as contagens de exclusão.


In [ ]:
arquivos = {
    "e10": e10_path,
    "e11": e11_path,
    "bt10": bt10_path,
    "bt11": bt11_path,
}
dados = {}
grade = None
for nome, caminho in arquivos.items():
    with rasterio.open(caminho) as src:
        grade_atual = (src.shape, src.crs, src.transform)
        if grade is None:
            grade = grade_atual
            perfil = src.profile.copy()
        elif grade_atual != grade:
            raise ValueError(f"A imagem {nome} não está na mesma grade das outras entradas.")
        dados[nome] = src.read(1, masked=True).astype(float).filled(np.nan)
e10 = dados["e10"]
e11 = dados["e11"]
bt10 = dados["bt10"]
bt11 = dados["bt11"]
mascara = ~np.isfinite(e10) | ~np.isfinite(e11) | ~np.isfinite(bt10) | ~np.isfinite(bt11)
# Ler QA na grade da primeira imagem e combinar com nodata antes do método.
mascara_qa = read_quality_mask(
    qa_pixel_path, qa_radsat_path,
    reference_path=next(iter(arquivos.values())),
    exclude_water=False,
)
mascara |= mascara_qa

print("Dimensões:", mascara.shape)
print("Pixels excluídos por QA:", int(mascara_qa.sum()))
print("Pixels excluídos por QA ou nodata:", int(mascara.sum()), "de", mascara.size)


## Passo 3 — Calcular a temperatura

Execute a chamada abaixo. As variáveis são matrizes carregadas no passo 2; os caminhos de arquivo foram usados no passo 1. `mask=mascara` aplica a seleção QA/nodata ao cálculo.


In [ ]:
resultado = split_window(
    lst_method='du-2015',
    emissivity_10=e10,
    emissivity_11=e11,
    brightness_band_10=bt10,
    brightness_band_11=bt11,
    mask=mascara,
    unit=unidade,
    return_diagnostics=diagnosticos,
)
imagem = resultado["lst"] if diagnosticos else resultado

## Passo 4 — Salvar e usar a saída

Execute esta célula para gravar o GeoTIFF em `saida`, preservando a grade. Confira o caminho absoluto e a quantidade de pixels válidos impressos. Pixels inválidos são gravados como NaN.

**Próximo uso:** Comparar a LST com a referência de campo; registrar se foram usados os coeficientes gerais ou estratificados por PWV.


In [ ]:
saida.mkdir(parents=True, exist_ok=True)
perfil.update(driver="GTiff", dtype="float32", count=1, nodata=np.nan, compress="deflate")
with rasterio.open(saida / "split_window__du-2015.tif", "w", **perfil) as dst:
    dst.write(np.asarray(imagem, dtype="float32"), 1)
    dst.update_tags(units=unidade, method='du-2015')
print("Pixels válidos:", int(np.isfinite(imagem).sum()), "de", imagem.size)
print("Saída:", saida.resolve())

## Como conferir a execução

Abra o GeoTIFF salvo em um SIG e confira a posição, a unidade e a banda informadas acima. Compare a contagem de pixels válidos com a área disponível: QA pode excluir pixels, e os critérios do próprio método podem produzir lacunas adicionais.

| Se acontecer | O que conferir |
| --- | --- |
| `FileNotFoundError` no passo 1 | Preencha os caminhos listados no erro, inclusive MTL quando solicitado e ambas as QA. |
| Erro de grade no passo 2 | Prepare arquivos com a mesma projeção, transformação, dimensões e recorte. Não redimensione as QA por interpolação contínua. |
| Erro de domínio no passo 3 | Confira a grandeza/unidade e os requisitos do método. |
| Saída com NaN | Confira as contagens QA/nodata e o suporte do método; NaN não deve ser substituído por temperatura inventada. |

[Voltar ao índice e à ordem de execução](README.md).


## Consulta — opções e detalhes do método

O roteiro executável termina acima. Leia os tópicos abaixo para adaptar a entrada ou consultar parâmetros; não execute todas as alternativas em sequência sobre a mesma saída.


### Entradas da função e opções

### Entradas do caminho principal

| Entrada da função | O que fornecer | Unidade / domínio |
|---|---|---|
| `emissivity_10` | Emissividade pronta, específica de B10 | Adimensional, (0,1] |
| `emissivity_11` | Emissividade pronta, específica de B11 | Adimensional, (0,1] |
| `brightness_band_10` | Temperatura de brilho TOA B10 | K |
| `brightness_band_11` | Temperatura de brilho TOA B11 | K |

**Com emissividade pronta:** omitir `emissivity_method` e as bandas usadas somente para calculá-la. A alternativa por bandas aparece na consulta de alternativas. Não combinar as duas rotas.

Todas as imagens usadas juntas devem ter a mesma grade: projeção, resolução, posição dos pixels e dimensões. As variáveis da chamada são arrays de pixels; não são caminhos de arquivo.

### O que é opcional

- **`mask`:** máscara booleana alinhada: True exclui o pixel e False permite avaliar. Opcional na API; neste exemplo é preenchida no passo 2 com QA e nodata. Não passar QA_PIXEL bruta.
- **`unit`:** saída em kelvin (padrão) ou Celsius; BT entra sempre em K.
- **`return_diagnostics`:** False retorna array; True retorna LST e intermediários.
- **`water_vapor`:** PWV escalar/raster em [0; 6,3] g/cm²; None usa coeficientes gerais e não estima PWV.

### Alternativas de uso

Escolha a alternativa que corresponde aos dados disponíveis; não execute todas as chamadas em sequência sobre a mesma saída.

### 5.1. Fornecer bandas para calcular a emissividade

Omitir as emissividades prontas. Vermelho/NIR devem ter a mesma grandeza e ser compatíveis com o modelo: radiância TOA neste exemplo, salvo Kerr, que usa reflectância superficial. O padrão interno de Split-window é Rongali; Avdan é bloqueado nessa rota.

```python
imagem = split_window(
    lst_method='du-2015',
    emissivity_method='rongali-2018',
    band_4_red=red,
    band_5_nir=nir,
    optical_quantity='radiance',
    brightness_band_10=bt10,
    brightness_band_11=bt11,
    mask=mascara,
)
```
### 5.2. Condicionar os coeficientes por PWV

Acrescentar `water_vapor=pwv` à chamada principal. Usar escalar ou raster alinhado em **[0; 6,3] g/cm²**. None mantém os coeficientes gerais, sem calcular vapor automaticamente. Para o workflow completo, consulte [Du com PWV](../docs/du-pwv-workflow.md).

### Saída e diagnósticos

`unit="celsius"` muda apenas a saída. `return_diagnostics=True` retorna dicionário; salvar resultado["lst"]. Com emissividade pronta, ndvi é None, exceto quando Kerr o usa. A máscara final incorpora as rejeições por domínio.

### Referência completa dos parâmetros

### Caminho principal

A chamada completa e a tabela abaixo seguem a mesma ordem.

```python
resultado = split_window(
    lst_method='du-2015',
    emissivity_10=e10,
    emissivity_11=e11,
    brightness_band_10=bt10,
    brightness_band_11=bt11,
    mask=mascara,
    unit=unidade,
    return_diagnostics=diagnosticos,
)
```

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `lst_method` | Obrigatório: selecionar o método | `None` (precisa informar) | — (seletor) | Usar `"du-2015"` neste notebook. |
| `emissivity_10` | Obrigatório nesta rota | `None` | Adimensional, (0,1] | Raster pronto de emissividade B10, na mesma grade das BT. Não recalculado pela função de LST. Omitir emissivity_method e bandas ópticas usadas apenas para emissividade. |
| `emissivity_11` | Obrigatório nesta rota | `None` | Adimensional, (0,1] | Raster pronto de emissividade B11, na mesma grade das BT. Não recalculado pela função de LST. Omitir emissivity_method e bandas ópticas usadas apenas para emissividade. |
| `brightness_band_10` | Obrigatório | Sem padrão | K (kelvin) | Raster de temperatura de brilho **TOA B10**, calibrado pelo MTL. Não fornecer DN, radiância, LST ou Celsius. |
| `brightness_band_11` | Obrigatório | Sem padrão | K (kelvin) | Raster de temperatura de brilho **TOA B11**, com a mesma forma e grade de B10. |
| `mask` | Opcional na API; fornecido neste exemplo | `None` | Booleano | Máscara de exclusão alinhada: True rejeita, False permite avaliar. O passo 2 lê ambas as QA e combina seus bits com nodata. Água segue a escolha exclude_water explícita neste notebook. Não fornecer QA bruta; a função pode acrescentar rejeições por domínio. |
| `unit` | Opcional | `"kelvin"` | K ou °C | `"kelvin"` ou `"celsius"`. Afeta apenas a saída; BT10/BT11 continuam entrando em K. A chamada acima escolhe Celsius explicitamente. |
| `return_diagnostics` | Opcional | `False` | — (booleano) | `False`: retorna somente o raster de LST. `True`: retorna dicionário com `lst`, `ndvi`, `emissivity_10`, `emissivity_11`, `invalid_mask`, `units` e `optical_contract`. `invalid_mask` é a rejeição final do cálculo, que pode incluir mais pixels que a máscara de entrada. |

### Parâmetros das alternativas

As entradas substituídas e as combinações permitidas são explicadas na consulta de alternativas. Os demais parâmetros permanecem os da tabela principal.

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `emissivity_method` | Opcional somente na rota por bandas | `None` → `"rongali-2018"` | Seletor | "rongali-2018"; "avdan-2016" é bloqueado nessa rota. Omitir com emissividade pronta; a seleção ocorre na etapa que gera a emissividade. |
| `band_4_red` | Obrigatório na rota por bandas; omitir com emissividade pronta | `None` | Neste exemplo: W m⁻² sr⁻¹ µm⁻¹; reflectância é adimensional | Banda vermelha calibrada. Deve ter a mesma convenção radiométrica do NIR e ser compatível com a parametrização da emissividade. O NDVI é calculado internamente. Com emissividades prontas, deixar `None`. |
| `band_5_nir` | Obrigatório na rota por bandas; omitir com emissividade pronta | `None` | Mesma grandeza e unidade do vermelho | Banda NIR calibrada e alinhada. Não combinar DN de uma banda com radiância/reflectância da outra. Com emissividades prontas, deixar `None`. |
| `optical_quantity` | Opcional; recomendado declarar | `None` (não declarada) | — (declaração de grandeza) | `"radiance"`, `"reflectance"`/`"toa_reflectance"` ou `"surface_reflectance"`. Apenas registra a convenção dos arrays; **não converte** radiância em reflectância nem faz correção atmosférica. O leitor deste exemplo oferece `"radiance"` ou `"reflectance"` (TOA). |
| `water_vapor` | Opcional | `None` | g/cm² | PWV escalar ou raster alinhado, entre **0 e 6,3**, inclusive. `None` usa coeficientes gerais e **não estima PWV**. Um valor informado seleciona as faixas de coeficientes; em sobreposições, calcula a média das estimativas de LST. Valores escalares inválidos geram erro; pixels inválidos de um raster viram NaN. |

Os valores escolhidos no exemplo podem diferir do padrão da API; a coluna de padrão descreve a função. As máscaras e as validações de domínio podem produzir NaN, sem preenchimento automático.

### Aplicabilidade e limites

Os coeficientes de recuperação são de Landsat 8; a aplicação a Landsat 9 é transferência experimental, separada da calibração radiométrica pelo MTL.

Execução e consistência numérica não demonstram acurácia independente em campo. A parametrização deve ser adequada à banda, sensor, superfície e condições da aplicação.

[Du et al. (2015) — artigo original](https://doi.org/10.3390/rs70100647).

Veja a [ficha metodológica](../docs/method-cards.md) para a equação, os coeficientes, o domínio e as adaptações. Execução numericamente correta não estabelece acurácia de campo. [Voltar ao índice](README.md).